In [3]:
import json
import os
annt_path='/home/v-jinjzhao/datasets/rec_human_celeb_ready/person_annts_val_ready.json'
with open(annt_path) as f:
    all_annts=json.load(f)
with open('output_human/pred.jsonl') as f:
    all_preds=[json.loads(line) for line in f]
print(len(all_annts),len(all_preds))

44738 44738


In [4]:
all_pred_bboxes=[]
all_gt_bboxes=[]
for annt,pred in zip(all_annts,all_preds):
    if(annt['recognition_category']=='celebrity'):
        gt_bbox=annt['bbox']
    else:
        x,y,w,h=annt['bbox']
        gt_bbox=[x,y,x+w,y+h]

    pred_bboxes=pred['bbox'][0]
    all_gt_bboxes.append(gt_bbox)
    all_pred_bboxes.append(pred_bboxes)

In [6]:
import json
import torch
from tqdm import tqdm
from overlaps import bbox_overlaps
# import average from math
from statistics import mean


def calculate_iou_acc(pred_bboxes,gt_bboxes, thresh=0.5):
    """
    pred_bboxes: [N,4]
    gt_bboxes: [N,4]
    calculate the iou and acc of the pred_bboxes and gt_bboxes,
    if iou(pred_bboxes[i],gt_bboxes[i])>0.5, then acc+=1
    all pred_bboxes_i and gt_bboxes_i are one to one assigned.
    
    """
    iou=bbox_overlaps(pred_bboxes,gt_bboxes,mode='iou', is_aligned=True)
    if(type(thresh) is not list):
        thresh=[thresh]
    accs=dict()
    for t in thresh:
        accs[t]=(iou>t).sum().item()/len(iou)
    return iou,accs




pred_bboxes=torch.tensor(all_pred_bboxes)
gt_bboxes=torch.tensor(all_gt_bboxes)
# create a list fron 0.5 to 0.9 with step 0.05
thresh=[i/100 for i in range(50,95,5)]
print(thresh)
iou,acc=calculate_iou_acc(pred_bboxes,gt_bboxes,thresh)
print_ths=[0.5,0.7,0.9]
for k,v in acc.items():
    if(k in print_ths):
        print(f"thresh={k}, acc={v}")
print(f"iou|0.5:0.9={mean([v for v in acc.values()])}")

print(iou)
print(acc)
print(f"{len(iou)}/{len(all_annts)}")

[0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
thresh=0.5, acc=0.6167910948187223
thresh=0.7, acc=0.5545844695784344
thresh=0.9, acc=0.4022978228798784
iou|0.5:0.9=0.5385776943289572
tensor([0.9354, 0.9354, 0.2907,  ..., 0.0682, 0.9418, 0.1143])
{0.5: 0.6167910948187223, 0.55: 0.6031561536054361, 0.6: 0.5898117931065313, 0.65: 0.573941615628772, 0.7: 0.5545844695784344, 0.75: 0.5335061916044526, 0.8: 0.5055433859358934, 0.85: 0.46756672180249453, 0.9: 0.4022978228798784}
44738/44738


In [7]:
for idx,annt in enumerate(all_annts):
    annt['PSALM_pred']=dict(
        pred_bboxes=pred_bboxes[idx].tolist(),
        iou=iou[idx].item()
    )


In [8]:
all_annts[-1]

{'annt_id': '0044737',
 'bbox': [196.128, 153.90599999999998, 840.3539999999999, 667.38],
 'bbox_area': 330793.301124,
 'caption': 'Zack Wheeler',
 'category_id': 1,
 'file_name': 'Zack_Wheeler_0180.jpg',
 'height': 681,
 'image_id': 'Zack_Wheeler_0180',
 'recognition_category': 'celebrity',
 'width': 1024,
 'source': 'web',
 'original_subset': 'val',
 'is_rewrite': False,
 'groundingGPT_pred': {'pred_bboxes': [256.0,
   115.72000122070312,
   778.239990234375,
   669.0],
  'iou': 0.46317657828330994},
 'shikra7b_pred': {'iou': 0.7647262811660767,
  'pred_bbox': [261.1199951171875,
   128.00799560546875,
   785.4080200195312,
   681.9920043945312]},
 'Lenna_pred': {'pred_bboxes': [170.35012817382812,
   126.15596771240234,
   758.0120849609375,
   672.5585327148438],
  'iou': 0.7939690351486206},
 'PSALM_pred': {'pred_bboxes': [164, 137, 335, 436],
  'iou': 0.11429689824581146}}

In [9]:
with open(annt_path,'w') as f:
    json.dump(all_annts,f,indent=4)